In [1]:
from catboost import CatBoostRegressor
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [2]:
df = pd.read_excel('task_1_2.xlsx')

In [3]:
from catboost import CatBoostRegressor
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error

df["Date"] = pd.to_datetime(df["Date"])
df = df.sort_values("Date")
df.set_index("Date", inplace=True)
# 1. Подготовка признаков (Feature Engineering)
data = pd.DataFrame()
data["y"] = df["series1"]

# Добавляем лаги (предыдущие значения ряда)
for lag in [1, 2, 3, 7, 14, 30, 365]:
    data[f"lag_{lag}"] = data["y"].shift(lag)

# Добавляем скользящие средние и стандартное отклонение
data["rolling_mean_7"] = data["y"].shift(1).rolling(window=7).mean()
data["rolling_std_7"] = data["y"].shift(1).rolling(window=7).std()
data["rolling_mean_30"] = data["y"].shift(1).rolling(window=30).mean()

# Извлекаем календарные признаки из даты
data["dayofweek"] = data.index.dayofweek
data["month"] = data.index.month
data["day"] = data.index.day
data["dayofyear"] = data.index.dayofyear
data["is_weekend"] = (data.index.dayofweek >= 5).astype(int)



In [4]:
# Удаляем строки с NaN, образовавшиеся из-за лагов
data = data.dropna()

# 2. Разделение на обучающую и тестовую выборки (последние 90 дней — тест)
train_data = data.iloc[:-90]
test_data = data.iloc[-90:]

X_train, y_train = train_data.drop(columns=["y"]), train_data["y"]
X_test, y_test = test_data.drop(columns=["y"]), test_data["y"]

# 3. Инициализация и обучение CatBoost
cb_model = CatBoostRegressor(
    iterations=1000, learning_rate=0.03, depth=9, loss_function="RMSE", verbose=100
)

cb_model.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=50)

# 4. Прогноз и оценка качества
predictions = cb_model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
mape = np.mean(np.abs((y_test - predictions) / y_test)) * 100

print(f"\nCatBoost MAE: {mae:.2f}")
print(f"CatBoost RMSE: {rmse:.2f}")
print(f"CatBoost MAPE: {mape:.2f}%")

0:	learn: 1274.4794591	test: 677.9062780	best: 677.9062780 (0)	total: 62.2ms	remaining: 1m 2s
100:	learn: 499.0339102	test: 460.7894479	best: 460.6785229 (98)	total: 252ms	remaining: 2.24s
Stopped by overfitting detector  (50 iterations wait)

bestTest = 460.5911335
bestIteration = 108

Shrink model to first 109 iterations.

CatBoost MAE: 349.50
CatBoost RMSE: 460.59
CatBoost MAPE: 10.62%


In [5]:
data.head()

,y,lag_1,lag_2,lag_3,lag_7,lag_14,lag_30,lag_365,rolling_mean_7,rolling_std_7,rolling_mean_30,dayofweek,month,day,dayofyear,is_weekend
Date,,,,,,,,,,,,,,,,
2016-01-01,1302.787781,8361.528849,8172.032444,7872.786039,5788.522981,4362.365145,3059.577364,1006.699649,6948.427089,1176.365831,4680.929656,4,1,1,1,0
2016-01-02,3378.365586,1302.787781,8361.528849,8172.032444,5549.086378,4243.929892,3099.055782,3197.751826,6307.607775,2448.003926,4622.370004,5,1,2,2,1
2016-01-03,3088.199217,3378.365586,1302.787781,8361.528849,6174.424513,4066.277013,3118.794991,3217.491035,5997.504805,2686.022383,4631.680330,6,1,3,3,1
2016-01-04,4635.555795,3088.199217,3378.365586,1302.787781,6720.608421,5031.129539,3553.057584,2151.573759,5556.615477,2897.135714,4630.660471,0,1,4,4,0
2016-01-05,4348.942483,4635.555795,3088.199217,3378.365586,7872.786039,5102.585475,2743.750024,4243.929892,5258.750816,2864.517991,4666.743745,1,1,5,5,0


In [6]:
pip install lightgbm


Note: you may need to restart the kernel to use updated packages.


In [7]:
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [9]:
# 1. Инициализация и обучение LightGBM
lgb_model = lgb.LGBMRegressor(
    n_estimators=1000,
    learning_rate=0.03,
    max_depth=9,
    random_state=42,
    verbose=-1,
)

# Обучаем модель, передавая eval_X и eval_y отдельно
callbacks = [lgb.early_stopping(stopping_rounds=50, verbose=False)]

lgb_model.fit(
    X_train,
    y_train,
    eval_X=X_test,  # Передаем признаки теста
    eval_y=y_test,  # Передаем целевую переменную теста
    callbacks=callbacks,
)

# 2. Прогноз и оценка качества
lgb_predictions = lgb_model.predict(X_test)

lgb_mae = mean_absolute_error(y_test, lgb_predictions)
lgb_rmse = np.sqrt(mean_squared_error(y_test, lgb_predictions))
lgb_mape = np.mean(np.abs((y_test - lgb_predictions) / y_test)) * 100

print(f"LightGBM MAE: {lgb_mae:.2f}")
print(f"LightGBM RMSE: {lgb_rmse:.2f}")
print(f"LightGBM MAPE: {lgb_mape:.2f}%")

LightGBM MAE: 363.43
LightGBM RMSE: 481.85
LightGBM MAPE: 11.17%
